In [62]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from scipy.stats import chi2_contingency, fisher_exact


1.Load data & Data Overview

In [63]:
df = pd.read_excel("../data/2022Cdata.xlsx")

In [64]:
print(df.shape)
df.info()
df.head(10)

(58, 5)
<class 'pandas.DataFrame'>
RangeIndex: 58 entries, 0 to 57
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   文物编号    58 non-null     int64
 1   纹饰      58 non-null     str  
 2   类型      58 non-null     str  
 3   颜色      54 non-null     str  
 4   表面风化    58 non-null     str  
dtypes: int64(1), str(4)
memory usage: 2.4 KB


,文物编号,纹饰,类型,颜色,表面风化
0,1,C,高钾,蓝绿,无风化
1,2,A,铅钡,浅蓝,风化
2,3,A,高钾,蓝绿,无风化
3,4,A,高钾,蓝绿,无风化
4,5,A,高钾,蓝绿,无风化
5,6,A,高钾,蓝绿,无风化
6,7,B,高钾,蓝绿,风化
7,8,C,铅钡,紫,风化
8,9,B,高钾,蓝绿,风化
9,10,B,高钾,蓝绿,风化


2. Missing Value

In [65]:
df.isnull().sum()

文物编号    0
纹饰      0
类型      0
颜色      4
表面风化    0
dtype: int64

3. Value counts

In [66]:
for col in ["类型","纹饰","颜色","表面风化"]:

        print("="*20)
        print(col)
        print(df[col].value_counts(dropna=False))
        

类型
类型
铅钡    40
高钾    18
Name: count, dtype: int64
纹饰
纹饰
C    30
A    22
B     6
Name: count, dtype: int64
颜色
颜色
浅蓝     20
蓝绿     15
深绿      7
紫       4
NaN     4
浅绿      3
深蓝      2
黑       2
绿       1
Name: count, dtype: int64
表面风化
表面风化
风化     34
无风化    24
Name: count, dtype: int64


In [67]:
print(df.columns)

Index(['文物编号', '纹饰', '类型', '颜色', '表面风化'], dtype='str')


## 1. Cross-tabulation between Glass Type and Weathering

Purpose:
To summarize the frequency distribution of weathering status under different glass types.

In [82]:
table_type_display=pd.crosstab( 
    df["类型"], df["表面风化"] , margins=True)
table_type

表面风化,无风化,风化
类型,,
铅钡,12,28
高钾,12,6


In [69]:
table_type=pd.crosstab( 
    df["类型"], df["表面风化"] )
table_type

表面风化,无风化,风化
类型,,
铅钡,12,28
高钾,12,6


### Chi-square Test

In [71]:
result=chi2_contingency(
    table_type, 
    correction=False)
result


Chi2ContingencyResult(statistic=np.float64(6.8803921568627455), pvalue=np.float64(0.008714644061182652), dof=1, expected_freq=array([[16.55172414, 23.44827586],
       [ 7.44827586, 10.55172414]]))

In [72]:
print("Chi-square stastistic :", result.statistic )
print("p-value:",result.pvalue)
print( "degree of freedom:" , result.dof)
print("expected frequency:",result.expected_freq)

Chi-square stastistic : 6.8803921568627455
p-value: 0.008714644061182652
degree of freedom: 1
expected frequency: [[16.55172414 23.44827586]
 [ 7.44827586 10.55172414]]


### expected cross tab under H0

In [73]:
expected = pd.DataFrame(
    result.expected_freq,
    index=table_type.index,
    columns=table_type.columns
)

expected

表面风化,无风化,风化
类型,,
铅钡,16.551724,23.448276
高钾,7.448276,10.551724


In [74]:
print("Minimum expected frequency:", expected.min().min())


Minimum expected frequency: 7.448275862068965


### Chi-square Assumption Check

All expected frequencies are greater than 5.

Therefore, the assumptions of the Pearson Chi-square test are satisfied.

Since p-value < 0.05,

the null hypothesis of independence is rejected.

Therefore,

glass type and weathering status are significantly associated.

## Cramer's V Test

In [75]:
type_prop = pd.crosstab (df["类型"], df["表面风化"], normalize="index")
type_prop

表面风化,无风化,风化
类型,,
铅钡,0.300000,0.700000
高钾,0.666667,0.333333


In [77]:
n = table_type.to_numpy().sum()
r,c = table_type.shape

cramers_v = np.sqrt(result.statistic /(n * min(r-1,c-1)))
cramers_v

np.float64(0.3444233600968322)

fisher exact test

In [79]:
from scipy.stats import fisher_exact
fisher_result = fisher_exact(table_type)
fisher_result

SignificanceResult(statistic=np.float64(0.21428571428571427), pvalue=np.float64(0.011327604477116615))

# Relationship 2

## Decoration vs weathering

### Cross Table

In [81]:
table_decoration = pd.crosstab (df["纹饰"], df["表面风化"])
table_decoration

表面风化,无风化,风化
纹饰,,
A,11,11
B,0,6
C,13,17


### Chi-square Tset

In [90]:
decoration_result = chi2_contingency (table_decoration, correction=False)
print(decoration_result.pvalue)
print(decoration_result.expected_freq)

0.08388839673210008
[[ 9.10344828 12.89655172]
 [ 2.48275862  3.51724138]
 [12.4137931  17.5862069 ]]


Since %33 of the data < 5 , chi-square may not be accurate.

So we try Monte Carlo chi-square.

### Monte Carlo Chi-square

In [85]:
from scipy.stats import chi2_contingency, MonteCarloMethod


In [86]:
mc_method = MonteCarloMethod(n_resamples=100000)

decoration_mc_result = chi2_contingency(table_decoration, correction=False, method=mc_method)
decoration_mc_result

Chi2ContingencyResult(statistic=np.float64(4.9565359477124185), pvalue=np.float64(0.1024089759102409), dof=nan, expected_freq=array([[ 9.10344828, 12.89655172],
       [ 2.48275862,  3.51724138],
       [12.4137931 , 17.5862069 ]]))

In [87]:
print ("Monte Carlo p-value:", decoration_mc_result.pvalue)

Monte Carlo p-value: 0.1024089759102409


p-value > 0.05 , decoration and weathering are independent and unrelated.

### Cramers' V

In [88]:
n = table_decoration.to_numpy().sum()
r,c  = table_decoration.shape
decoration_cramers_v = np.sqrt( decoration_result.statistic/(n*min(n-1,c-1)))
decoration_cramers_v



np.float64(0.29233117579189066)

“Cramér’s V 为 0.29，表明样本中存在弱至接近中等程度的关联趋势。然而 Monte Carlo 检验 p\approx0.10>0.05，故该关联未达到统计显著水平。”
对纹饰与表面风化建立 3\times2 列联表后发现，部分单元格理论频数小于5，普通 Pearson 卡方检验的渐近近似条件不够理想，因此进一步采用 Monte Carlo 方法进行显著性检验。Monte Carlo 检验得到 p\approx0.10>0.05，故在5%的显著性水平下，尚无充分证据认为纹饰与表面风化之间存在显著统计关联。另一方面，Cramér’s V 约为0.29，表明样本数据中仍呈现一定程度的关联趋势，其中B类纹饰的6件文物均发生风化；但由于B类样本量较小，该现象的稳定性仍需更多样本验证。因此，本文不将纹饰视为表面风化的显著相关因素。